In [29]:
import json
import os

try:
    with open('../conf/config.json', 'r') as f:
        config = json.load(f)
    storage_path = config['storage_path']
    data_path = config['data_path']
    derivatives_path = config['derivatives_path']
    fsl_path = config['fsl_path']
    project_path = config['project_path']
    assert os.path.exists(storage_path), "The specified storage path does not exist."
    assert os.path.exists(data_path), "The specified data path does not exist."
    assert os.path.exists(derivatives_path), "The specified derivatives path does not exist."
    assert os.path.exists(fsl_path), "The specified FSL path does not exist."
    assert os.path.exists(project_path), "The specified project path does not exist."

except FileNotFoundError:
    raise FileNotFoundError("config.json file not found. Please create it with the required paths.")

import sys
sys.path.append(f'{project_path}/models/generative_models')
sys.path.append(f'{project_path}/models')


In [30]:
def is_interactive():
    import __main__ as main
    return not hasattr(main, '__file__')

In [ ]:
import argparse

parser = argparse.ArgumentParser(description="Model Training Configuration")
parser.add_argument("--model_name", type=str, help="Name of the model to use")
parser.add_argument("--seed", type=int, default=0, help="random seed")
parser.add_argument("--base_dirname", type=str, default="sub-005_ses-03_task-C_recons_byrepeats_rt", help='base dir path')

if is_interactive():
    jupyter_args = "--model_name sub-005-ses-01_task-C-rt_ft_split=repeats3_delay=0_epochs=150_delay=0 --seed 0"
    parser = parser.parse_args(jupyter_args.split())
else:
    args = parser.parse_args()

# create global variables without the args prefix
for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name)

In [ ]:
# configure paths
base_dir = os.path.join(derivatives_path, base_dirname)
# model_name = "sub-005-ses-01_task-C-rt_ft_split=repeats3_delay=63_epochs=150/"

dump_dir = os.path.join(base_dir, model_name, f"{seed}")

In [40]:
print(dump_dir)

/scratch/am10150/projects/rtcloud-projects/mindeye/3t/derivatives/sub-005_ses-03_task-C_recons_byrepeats_rt/sub-005-ses-01_task-C-rt_ft_split=repeats3_delay=63_epochs=150_delay=63


In [41]:
from collections import defaultdict
import torch
from tqdm import tqdm

def collect_tensors(dump_dir: str):
    data = defaultdict(list)
    # go over each directory and concat recons
    folders = sorted(os.listdir(dump_dir))
    for folder in tqdm(folders):
        if 'sanity_check_individual_reps' in folder \
            or not os.path.isdir(os.path.join(dump_dir, folder)) \
            or folder.startswith("."):
            continue
    
        clip_voxels = torch.load(os.path.join(dump_dir, folder, "all_clipvoxels.pt")).unsqueeze(0)
        gt = torch.load(os.path.join(dump_dir, folder, "all_ground_truth.pt")).unsqueeze(0)
        recons = torch.load(os.path.join(dump_dir, folder, "all_recons.pt")).unsqueeze(0)
        retrieved = torch.load(os.path.join(dump_dir, folder, "all_retrieved.pt")).unsqueeze(0)
        
        data['clip_voxels'].append(clip_voxels)
        data['gt'].append(gt)
        data['recons'].append(recons)
        data['retrieved'].append(retrieved)
    
    for key in data.keys():
        data[key] = torch.cat(data[key], dim=0)

    return data

In [42]:
data = collect_tensors(dump_dir)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:08<00:00,  6.14it/s]


In [43]:
# num_images x repeats x channel x height x width
data['recons'].shape

torch.Size([50, 3, 3, 224, 224])

In [44]:
data['gt'].shape

torch.Size([50, 3, 3, 224, 224])

In [45]:
from utils_mindeye import calculate_retrieval_metrics, calculate_alexnet, calculate_clip, calculate_swav, calculate_efficientnet_b1, calculate_inception_v3, calculate_pixcorr, calculate_ssim

def calc_metrics(
    gt,
    clip_voxels,
    recons
):
    assert gt.shape[0] == recons.shape[0] and clip_voxels.shape[0] == gt.shape[0], \
        "batch dim should be same across all gt, clip_voxels and recons"

    device = 'cuda' if torch.cuda.is_available() else 'cpu'    
    gt = gt.to(torch.float16).to(device)
    clip_voxels = clip_voxels.to(torch.float16).to(device)
    recons = recons.to(torch.float16).to(device)

    print(gt.shape, clip_voxels.shape, recons.shape)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        print('retrieval metrics')
        all_fwd_acc, all_bwd_acc = calculate_retrieval_metrics(
            clip_voxels, gt
        )
        print("pixcor..")
        pixcorr = calculate_pixcorr(recons, gt)
        
        print("ssim..")
        ssim_ = calculate_ssim(recons, gt)
        
        print("alexnet..")
        alexnet2, alexnet5 = calculate_alexnet(recons, gt)

        print("inception..")
        inception = calculate_inception_v3(recons, gt)

        print("clip..")
        clip_ = calculate_clip(recons, gt)

        print("efficientnet..")
        efficientnet = calculate_efficientnet_b1(recons, gt)

        print("swav..")
        swav = calculate_swav(recons, gt)

    return {
        'all_fwd_acc': all_fwd_acc,
        'all_bwd_acc': all_bwd_acc,
        'pixcorr': pixcorr,
        'ssim': ssim_,
        'alexnet2': alexnet2,
        'alexnet5': alexnet5,
        'inception': inception,
        'clip': clip_,
        'efficientnet': efficientnet,
        'swav': swav,
        
    }


In [46]:
metrics = []
for repeat in tqdm(range(3)):
    metric = calc_metrics(
        gt=data['gt'][:, repeat],
        clip_voxels=data['clip_voxels'][:, repeat],
        recons=data['recons'][:, repeat]
    )
    metrics.append(metric)

  0%|                                                                                                                           | 0/3 [00:00<?, ?it/s]

torch.Size([50, 3, 224, 224]) torch.Size([50, 1, 256, 1664]) torch.Size([50, 3, 224, 224])
retrieval metrics
Loading clip_img_embedder
The total pool of images and clip voxels to do retrieval on is:  50
Creating embeddings for images
Calculating retrieval metrics


/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


overall fwd percent_correct: 0.7000
overall bwd percent_correct: 0.6400
pixcor..
torch.Size([50, 541875])
torch.Size([50, 541875])



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 95.97it/s]


...

Pixel Correlation: 0.12406555967400477
ssim..
converted, now calculating ssim...



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 59.75it/s]


SSIM: 0.3380585212394361
alexnet..
Loading AlexNet

---early, AlexNet(2)---
2-way Percent Correct (early AlexNet): 0.8816

---mid, AlexNet(5)---
2-way Percent Correct (mid AlexNet): 0.8963
inception..
Loading Inception V3
2-way Percent Correct (Inception V3): 0.7722
clip..
Loading CLIP
2-way Percent Correct (CLIP): 0.6963
efficientnet..
Loading EfficientNet B1
Distance EfficientNet B1: 0.8970532966179005
swav..
Loading SwAV


Using cache found in /scratch/am10150/.cache/torch/hub/facebookresearch_swav_main
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [02:48<00:00, 56.30s/it]

Distance SwAV: 0.5311394077931467


In [47]:
metrics_avg = defaultdict(list)
for metric in metrics:
    for key, value in metric.items():
        metrics_avg[key].append(value)
metrics_avg['rep'] = [1,2,3]

#### Per avg metrics

In [48]:
import pandas as pd
avg_metrics_df = pd.DataFrame(metrics_avg)

In [49]:
print(avg_metrics_df)

   all_fwd_acc  all_bwd_acc   pixcorr      ssim  alexnet2  alexnet5  \
0         0.70         0.64  0.083549  0.341915  0.767755  0.761633   
1         0.80         0.76  0.073416  0.330583  0.824490  0.844490   
2         0.84         0.80  0.124066  0.338059  0.881633  0.896327   

   inception      clip  efficientnet      swav  rep  
0   0.720816  0.615510      0.937285  0.569789    1  
1   0.755102  0.649388      0.906846  0.553219    2  
2   0.772245  0.696327      0.897053  0.531139    3  


### Computing metrics across individual repeats

In [50]:
data_repeats = collect_tensors(os.path.join(dump_dir, "sanity_check_individual_reps"))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:08<00:00,  6.07it/s]


In [51]:
data_repeats["gt"].shape

torch.Size([50, 3, 3, 224, 224])

In [52]:
metrics = []
for repeat in tqdm(range(3)):
    metric = calc_metrics(
        gt=data_repeats['gt'][:, repeat],
        clip_voxels=data_repeats['clip_voxels'][:, repeat],
        recons=data_repeats['recons'][:, repeat]
    )
    metrics.append(metric)

  0%|                                                                                                                           | 0/3 [00:00<?, ?it/s]

torch.Size([50, 3, 224, 224]) torch.Size([50, 1, 256, 1664]) torch.Size([50, 3, 224, 224])
retrieval metrics
Loading clip_img_embedder
The total pool of images and clip voxels to do retrieval on is:  50
Creating embeddings for images
Calculating retrieval metrics


/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


overall fwd percent_correct: 0.7000
overall bwd percent_correct: 0.6400
pixcor..
torch.Size([50, 541875])
torch.Size([50, 541875])



100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 101.41it/s]


...

Pixel Correlation: 0.1138687149854471
ssim..
converted, now calculating ssim...



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 57.42it/s]


SSIM: 0.3562682665797706
alexnet..
Loading AlexNet

---early, AlexNet(2)---
2-way Percent Correct (early AlexNet): 0.8265

---mid, AlexNet(5)---
2-way Percent Correct (mid AlexNet): 0.8376
inception..
Loading Inception V3
2-way Percent Correct (Inception V3): 0.7086
clip..
Loading CLIP
2-way Percent Correct (CLIP): 0.6906
efficientnet..
Loading EfficientNet B1
Distance EfficientNet B1: 0.9068558212481164
swav..
Loading SwAV


Using cache found in /scratch/am10150/.cache/torch/hub/facebookresearch_swav_main
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [02:35<00:00, 51.88s/it]

Distance SwAV: 0.5402299326979412


In [53]:
metrics_rep = defaultdict(list)
for metric in metrics:
    for key, value in metric.items():
        metrics_rep[key].append(value)

metrics_rep['rep'] = [1,2,3]

In [54]:
import pandas as pd
rep_metrics_df = pd.DataFrame(metrics_rep)

In [55]:
rep_metrics_df

,all_fwd_acc,all_bwd_acc,pixcorr,ssim,alexnet2,alexnet5,inception,clip,efficientnet,swav,rep
0,0.70,0.64,0.059418,0.338663,0.741224,0.802449,0.671837,0.615102,0.928592,0.571471,1
1,0.70,0.68,0.080151,0.335132,0.813878,0.826939,0.676327,0.620408,0.912670,0.547483,2
2,0.76,0.66,0.113869,0.356268,0.826531,0.837551,0.708571,0.690612,0.906856,0.540230,3


### Combining all

In [56]:
avg_metrics_df['category'] = 'avg'
rep_metrics_df['category'] = 'rep'

combined_df = pd.concat([avg_metrics_df, rep_metrics_df])

In [57]:
metrics_save_path = os.path.join(dump_dir, 'metrics.csv')

In [58]:
# rearranging column order
cols_order = ['rep', 'category'] + list(metrics[0].keys())

combined_df[cols_order].to_csv(metrics_save_path, index=False)

In [59]:
combined_df[cols_order]

,rep,category,all_fwd_acc,all_bwd_acc,pixcorr,ssim,alexnet2,alexnet5,inception,clip,efficientnet,swav
0,1,avg,0.70,0.64,0.083549,0.341915,0.767755,0.761633,0.720816,0.615510,0.937285,0.569789
1,2,avg,0.80,0.76,0.073416,0.330583,0.824490,0.844490,0.755102,0.649388,0.906846,0.553219
2,3,avg,0.84,0.80,0.124066,0.338059,0.881633,0.896327,0.772245,0.696327,0.897053,0.531139
0,1,rep,0.70,0.64,0.059418,0.338663,0.741224,0.802449,0.671837,0.615102,0.928592,0.571471
1,2,rep,0.70,0.68,0.080151,0.335132,0.813878,0.826939,0.676327,0.620408,0.912670,0.547483
2,3,rep,0.76,0.66,0.113869,0.356268,0.826531,0.837551,0.708571,0.690612,0.906856,0.540230


In [60]:
combined_df[combined_df['category'] == 'rep'].describe()

,all_fwd_acc,all_bwd_acc,pixcorr,ssim,alexnet2,alexnet5,inception,clip,efficientnet,swav,rep
count,3.000000,3.00,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.0
mean,0.720000,0.66,0.084479,0.343354,0.793878,0.822313,0.685578,0.642041,0.916039,0.553061,2.0
std,0.034641,0.02,0.027482,0.011322,0.046036,0.018002,0.020039,0.042148,0.011253,0.016350,1.0
min,0.700000,0.64,0.059418,0.335132,0.741224,0.802449,0.671837,0.615102,0.906856,0.540230,1.0
25%,0.700000,0.65,0.069784,0.336897,0.777551,0.814694,0.674082,0.617755,0.909763,0.543857,1.5
50%,0.700000,0.66,0.080151,0.338663,0.813878,0.826939,0.676327,0.620408,0.912670,0.547483,2.0
75%,0.730000,0.67,0.097010,0.347466,0.820204,0.832245,0.692449,0.655510,0.920631,0.559477,2.5
max,0.760000,0.68,0.113869,0.356268,0.826531,0.837551,0.708571,0.690612,0.928592,0.571471,3.0


### Plotting reconstructions

In [61]:
import matplotlib.pyplot as plt
import numpy as np

def plot_side_by_side_images(ground_truth, averaged_recons, reps_recons, save_path=None):
    """
    Plots 7 images (ground_truth + 3 averaged_recons + 3 reps_recons) per row for each of 50 samples.
    
    Parameters:
    - ground_truth: shape (50, 3, 224, 224)
    - averaged_recons: shape (50, 3, 3, 224, 224)
    - reps_recons: shape (50, 3, 3, 224, 224)
    - save_path: optional, if given, saves the plot to this file, else shows interactively.
    """
    # Convert to numpy if torch Tensor
    if hasattr(ground_truth, 'detach'):
        ground_truth = ground_truth.detach().cpu().numpy()
    if hasattr(averaged_recons, 'detach'):
        averaged_recons = averaged_recons.detach().cpu().numpy()
    if hasattr(reps_recons, 'detach'):
        reps_recons = reps_recons.detach().cpu().numpy()
            
    num_samples = ground_truth.shape[0]
    fig, axes = plt.subplots(num_samples, 7, figsize=(21, 3 * num_samples))

    for idx in range(num_samples):
        # Collect images for current sample
        images = [ground_truth[idx]]
        images += list(averaged_recons[idx])  # three averaged recons
        images += list(reps_recons[idx])      # three reps recons

        for j, img in enumerate(images):
            # Transpose if shape is (3, H, W) to (H, W, 3)
            if img.shape[0] == 3:
                img = np.transpose(img, (1, 2, 0))
            axes[idx, j].imshow(np.clip(img, 0, 1))
            axes[idx, j].axis('off')
        
        if idx == 0:
            labels = ['GT', 'Avg_1', 'Avg_2', 'Avg_3', 'Rep_1', 'Rep_2', 'Rep_3']
            for j in range(7):
                axes[0, j].set_title(labels[j])
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


In [62]:
recons_save_path = os.path.join(dump_dir, 'recons.png')
plot_side_by_side_images(data['gt'][:, 0], data['recons'], data_repeats['recons'], save_path=recons_save_path)